In [ ]:
from transformers import AutoTokenizer

/home/jovyan/Works/MSc_Thesis/mh_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [3]:
def prepare_dataset(dataset):
   
    
    # Data prompt
    data_prompt = """You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.

### Input:
Patient ID: {}
Pateint Query: {}

### Response:
{}"""

    # Formatting the prompt
    EOS_TOKEN = tokenizer.eos_token
    def formatting_prompt(examples):
        patient_ids = examples["patient_id"]
        queries = examples["patient_query"]
        responses = examples["assistant_response"]
        texts = []
        
        for patient_id, query, response in zip(patient_ids, queries, responses):
            text = data_prompt.format(patient_id, query, response) + EOS_TOKEN
            texts.append(text)
        
        return {"text": texts}
    
    dataset_with_text_column = dataset.map(formatting_prompt, batched=True)
    
    return dataset_with_text_column



In [4]:
# Load the dataset
from datasets import load_dataset
dataset_path = "/home/jovyan/Works/MSc_Thesis/dataset/mh_function_calling_dataset.csv"
data_file = {"train":dataset_path}
fc_dataset = load_dataset("csv",data_files=data_file)
fc_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days'],
        num_rows: 4000
    })
})

In [5]:
formatted_fc_dataset = prepare_dataset(fc_dataset)
formatted_fc_dataset

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'text'],
        num_rows: 4000
    })
})

In [6]:
# One sample
formatted_fc_dataset["train"][0]

{'patient_query': 'I want to track if movement helps my mood over 30 days',
 'assistant_response': '<think>\nDaily activity patterns reflected in step count are important indicators of mental health status. Social isolation, agoraphobia, or depression can significantly reduce daily movement. I should retrieve their step data for the specified number of days to assess their functional status.\n</think>\n\n<function>\n{\n  "name": "get_steps_data",\n  "parameters": {\n    "patient_id": "UID_920",`\n    "num_days": 30\n  }\n}\n</function>',
 'function_name': 'get_steps_data',
 'patient_id': 'UID_920',
 'num_days': 30,
 'text': 'You are a helpful mental health assistant. When users ask for health data, use the available functions to retrieve the information. Think through your response before choosing a function. Use <think>your reasoning</think> tags to show your thought process, then use <function>json</function> tags to call the appropriate function.\n\n### Input:\nPatient ID: UID_920\n

In [7]:
# Split the dataset into train and test (80% train, 20% test)
split_formatted_fc_dataset = formatted_fc_dataset["train"].train_test_split(train_size=0.8, seed=42)
split_formatted_fc_dataset 

DatasetDict({
    train: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'text'],
        num_rows: 3200
    })
    test: Dataset({
        features: ['patient_query', 'assistant_response', 'function_name', 'patient_id', 'num_days', 'text'],
        num_rows: 800
    })
})

In [8]:
# Save to disk
dataset_based_path = "/home/jovyan/Works/MSc_Thesis/dataset"
split_formatted_fc_dataset.save_to_disk(f"{dataset_based_path}/mh-function-calling-dataset-hf")

Saving the dataset (1/1 shards): 100%|██████████| 800/800 [00:00<00:00, 46285.17 examples/s]
